# EDA & Data Cleaning – Brent Crude Oil Analysis

**Five variables used in the dataset:**

| File Name | Symbol | Description | Frequency |
|------------|--------|-------------|------------|
| DCOILBRENTEU | OIL | Brent crude oil price (USD/barrel) | Daily |
| CPIAUCSL | CPI | U.S. Consumer Price Index | Monthly |
| DTWEXBGS | USD | U.S. Dollar Index (trade-weighted exchange rate) | Daily |
| FEDFUNDS | FED | U.S. Federal Funds Interest Rate (%) | Monthly |
| INDPRO | IND | U.S. Industrial Production Index | Monthly |

## 0. Library setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': False, 
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 100,
})
PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

print('✅ Libraries imported successfully!')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')

## 1. Read raw data

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"D:\PYTHON\SUBJECTS\Timeseries\Final-Timeseries\Raw_data"
)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Directory not found: {DATA_DIR}"
    )

print("Files inside the directory:\n")

for f in DATA_DIR.iterdir():
    print(" -", f.name)

files = {
    'OIL': None,
    'CPI': None,
    'USD': None,
    'FED': None,
    'IND': None,
}

keywords = {
    'DCOILBRENTEU': 'OIL',
    'CPIAUCSL': 'CPI',
    'DTWEXBGS': 'USD',
    'FEDFUNDS': 'FED',
    'INDPRO': 'IND'
}

for file in DATA_DIR.glob("*.xlsx"):

    filename = file.stem.upper()

    for key, var in keywords.items():

        if key in filename:
            files[var] = file

missing = [k for k, v in files.items() if v is None]

if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}"
    )

col_map = {
    'DCOILBRENTEU': 'OIL',
    'CPIAUCSL': 'CPI',
    'DTWEXBGS': 'USD',
    'FEDFUNDS': 'FED',
    'INDPRO': 'IND'
}

raw = {}
monthly = {}

for name, filepath in files.items():

    print(f"\nReading file: {filepath.name}")

    df = pd.read_excel(
        filepath,
        parse_dates=['observation_date']
    )

    # Rename date column
    df.rename(
        columns={'observation_date': 'date'},
        inplace=True
    )

    # Set date as index
    df.set_index('date', inplace=True)

    # Rename feature columns
    df.columns = [
        col_map.get(c, c)
        for c in df.columns
    ]

    # Store raw dataframe
    raw[name] = df.copy()

    # Resample to monthly frequency
    monthly[name] = df.resample('ME').last()

    # Determine frequency label
    freq = 'Daily' if len(df) > 500 else 'Monthly'

    print(
        f'[{name}] {freq} | '
        f'{df.index[0].date()} → {df.index[-1].date()} | '
        f'{len(df)} observations | '
        f'NaN values: {df.isnull().sum().values[0]}'
    )



## 2. Merge data & clean

In [ ]:
# Concatenate all monthly dataframes
df_raw = pd.concat(monthly.values(), axis=1)

print('=== BEFORE DATA CLEANING ===')

print(f'Shape: {df_raw.shape}')

print(
    f'Time range: '
    f'{df_raw.index[0].date()} → {df_raw.index[-1].date()}'
)

print()

print('Missing values by column:')

print(
    df_raw.isnull().sum().to_frame('NaN_count').assign(
        Percentage=lambda x: (
            x['NaN_count'] / len(df_raw) * 100
        ).round(2)
    )
)

In [ ]:
# ── NaN cleaning: forward-fill then backward-fill (limit 3 months) ────────────
df_raw.ffill(limit=3, inplace=True)   # Fill NaN with previous value (max 3 months)
df_raw.bfill(limit=3, inplace=True)   # Fill remaining NaN with next value
df_raw.dropna(inplace=True)           # Drop rows still containing NaN

print('=== AFTER CLEANING ===')
print(f'Shape: {df_raw.shape}')
print(f'Time range: {df_raw.index[0].date()} → {df_raw.index[-1].date()}')
print(f'Remaining NaN count: {df_raw.isnull().sum().sum()}')
print()
df_raw.head()

## 3. Basic descriptive statistics

In [ ]:
# ── Extended statistics ─────────────────────────────────────────────────────────
def extended_stats(df):
    desc = df.describe().T
    desc['median'] = df.median()
    desc['skew']   = df.skew()
    desc['kurt']   = df.kurt()   # excess kurtosis (chuẩn = 0)
    desc['cv%']    = (desc['std'] / desc['mean'] * 100).round(2)  # coefficient of variation
    desc['range']  = desc['max'] - desc['min']
    return desc[['count', 'mean', 'median', 'std', 'cv%', 'min', '25%', '75%', 'max', 'skew', 'kurt', 'range']]

stats_df = extended_stats(df_raw)
print('EXTENDED DESCRIPTIVE STATISTICS')
print('='*90)
print(stats_df.round(4).to_string())
print()
print('Notes:')
print('  cv%  : Coefficient of variation (std/mean × 100) — higher means more variability')
print('  skew : Skewness (>0 right skew, <0 left skew, =0 symmetric)')
print('  kurt : Excess kurtosis (>0 heavier tails, <0 lighter tails)')

## 4. Time Series trends

In [ ]:
# Mark major historical events
events = {
    'GFC\n(2008)':    ('2008-09-01', '#e74c3c'),
    'OPEC\n(2014)':   ('2014-11-01', '#e67e22'),
    'COVID\n(2020)':  ('2020-03-01', '#8e44ad'),
    'Ukraine\n(2022)':('2022-02-01', '#2ecc71'),
}

var_labels = {
    'OIL': ('Brent Oil Price', 'USD/barrel', '#1f77b4'),
    'CPI': ('U.S. CPI', 'Index', '#ff7f0e'),
    'USD': ('USD Index', 'Points', '#2ca02c'),
    'FED': ('Fed Rate', '%/year', '#d62728'),
    'IND': ('Industrial Production', 'Index', '#9467bd'),
}

fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)
fig.suptitle('Economic variable trends over time (2006–2025)', 
             fontsize=15, fontweight='bold', y=1.01)

for ax, (col, (label, unit, color)) in zip(axes, var_labels.items()):
    ax.plot(df_raw.index, df_raw[col], color=color, lw=1.5, alpha=0.9)
    ax.fill_between(df_raw.index, df_raw[col], alpha=0.15, color=color)
    ax.set_ylabel(f'{label}\n({unit})', fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.1f}'))
    
    # Add event lines
    for ev_name, (ev_date, ev_color) in events.items():
        ax.axvline(pd.Timestamp(ev_date), color=ev_color, lw=1.2, 
                   ls='--', alpha=0.7)
        if ax == axes[0]:
            ax.text(pd.Timestamp(ev_date), ax.get_ylim()[1]*0.95, 
                    ev_name, fontsize=7.5, ha='center', color=ev_color,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Distribution of each variable (Histogram + KDE + Q-Q Plot)

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 18))
fig.suptitle('Probability distribution of each variable (Histogram + Q-Q Plot)', 
             fontsize=14, fontweight='bold')

for i, (col, (label, unit, color)) in enumerate(var_labels.items()):
    data = df_raw[col].dropna()
    
    # ── Histogram + KDE ──────────────────────────────────────────────────────
    ax1 = axes[i, 0]
    ax1.hist(data, bins=40, color=color, alpha=0.6, density=True, edgecolor='white', lw=0.5)
    
    # KDE thực tế
    kde_x = np.linspace(data.min(), data.max(), 300)
    kde = stats.gaussian_kde(data)
    ax1.plot(kde_x, kde(kde_x), color=color, lw=2, label='KDE thực tế')
    
    # Theoretical reference line
    mu, sigma = data.mean(), data.std()
    ax1.plot(kde_x, stats.norm.pdf(kde_x, mu, sigma), 
             'k--', lw=1.5, alpha=0.7, label='Normal distribution')
    
    # Mean and median lines
    ax1.axvline(mu, color='red', lw=1.5, ls='-', alpha=0.8, label=f'Mean={mu:.2f}')
    ax1.axvline(data.median(), color='orange', lw=1.5, ls='--', alpha=0.8, label=f'Median={data.median():.2f}')
    
    sk = data.skew()
    ku = data.kurt()
    ax1.set_title(f'{label}\nskew={sk:.3f}  kurt={ku:.3f}', fontsize=10)
    ax1.set_xlabel(unit); ax1.set_ylabel('Density')
    ax1.legend(fontsize=7, loc='upper right')
    
    # ── Q-Q Plot ─────────────────────────────────────────────────────────────
    ax2 = axes[i, 1]
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm')
    ax2.scatter(osm, osr, color=color, alpha=0.5, s=10)
    line_x = np.array([osm[0], osm[-1]])
    ax2.plot(line_x, slope * line_x + intercept, 'r-', lw=2, label=f'R²={r**2:.4f}')
    ax2.set_title(f'Q-Q Plot: {label}', fontsize=10)
    ax2.set_xlabel('Theoretical quantile (normal)')
    ax2.set_ylabel('Sample quantile')
    ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('\n💡 Interpreting Q-Q Plot:')
print('  - Points near the red line → distribution close to normal')
print('  - Points deviating in the tails → heavy tails or skewness')

## 6. Jarque-Bera normality test

In [ ]:
print('JARQUE-BERA TEST (H0: normal distribution)')
print('='*65)
print(f'{"Variable":<6} {"Skewness":>10} {"Kurtosis":>10} {"JB Stat":>12} {"p-value":>10} {"Conclusion":<20}')
print('-'*65)

for col, (label, _, _) in var_labels.items():
    data = df_raw[col].dropna()
    jb_stat, jb_p = stats.jarque_bera(data)
    sk, ku = data.skew(), data.kurt()
    conclusion = '❌ Non-normal' if jb_p < 0.05 else '✅ Probably normal'
    print(f'{col:<6} {sk:>10.4f} {ku:>10.4f} {jb_stat:>12.2f} {jb_p:>10.4f} {conclusion}')

print()
print('Meaning: p < 0.05 rejects H0 → non-normal distribution → consider log transform or non-parametric tests')

## 7. Outlier detection

In [ ]:
# ── Outlier detection with boxplot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(16, 6))
fig.suptitle('Outlier detection using Boxplot (IQR Method)', 
             fontsize=13, fontweight='bold')

for ax, (col, (label, unit, color)) in zip(axes, var_labels.items()):
    data = df_raw[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outlier = ((data < lower) | (data > upper)).sum()
    
    bp = ax.boxplot(data, patch_artist=True, notch=True,
                    boxprops=dict(facecolor=color, alpha=0.4),
                    medianprops=dict(color='red', lw=2),
                    flierprops=dict(marker='o', color='red', alpha=0.5, markersize=4))
    ax.set_title(f'{col}\n{label}', fontsize=10)
    ax.set_ylabel(unit, fontsize=9)
    ax.set_xticks([])
    ax.text(1, data.max()*0.98, f'{n_outlier} outliers\n({n_outlier/len(data)*100:.1f}%)',
            ha='center', fontsize=8, color='red',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# ── Outlier summary table ─────────────────────────────────────────────────────
print('OUTLIER SUMMARY TABLE (IQR ×1.5 method)')
print('='*70)
print(f'{"Variable":<6} {"Q1":>10} {"Q3":>10} {"IQR":>10} {"Lower bound":>14} {"Upper bound":>14} {"Outlier count":>10}')
print('-'*70)

for col in var_labels:
    data = df_raw[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((data < lower) | (data > upper)).sum()
    print(f'{col:<6} {Q1:>10.2f} {Q3:>10.2f} {IQR:>10.2f} {lower:>14.2f} {upper:>14.2f} {n_out:>10}')

In [ ]:
# ── Identify oil price outlier dates (OIL) ─────────────────────
data_oil = df_raw['OIL']
Q1, Q3 = data_oil.quantile(0.25), data_oil.quantile(0.75)
IQR = Q3 - Q1
oil_outliers = data_oil[(data_oil < Q1 - 1.5*IQR) | (data_oil > Q3 + 1.5*IQR)]

print(f'\n📌 OIL outlier months:')
print(oil_outliers.sort_values().to_frame().rename(columns={'OIL': 'Oil price (USD/barrel)'}).to_string())

## 8. Correlation matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (method, title) in zip(axes, [
    ('pearson', 'Pearson (linear)'),
    ('spearman', 'Spearman (rank/non-linear)'),
]):
    corr = df_raw.corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))  # only show lower triangle
    
    sns.heatmap(
        corr, mask=mask, ax=ax, annot=True, fmt='.3f',
        cmap='RdYlGn', center=0, vmin=-1, vmax=1,
        linewidths=0.5, linecolor='white',
        annot_kws={'size': 11, 'weight': 'bold'},
        square=True, cbar_kws={'label': 'Correlation coefficient'}
    )
    ax.set_title(f'Correlation {title}', fontsize=13, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.suptitle('Pearson vs Spearman correlation comparison', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💡 Interpretation:')
print('  |r| > 0.7 : Strong | 0.4–0.7 : Moderate | < 0.4 : Weak')
print('  Pearson measures linear correlation; Spearman measures monotonic correlation (captures non-linear relations)')

In [ ]:
# ── Scatter matrix ─────────────────────────────────────────────────────────────
from pandas.plotting import scatter_matrix

fig = plt.figure(figsize=(12, 10))
axes_sm = scatter_matrix(df_raw, alpha=0.3, figsize=(12, 10),
                         diagonal='kde', color='steelblue',
                         hist_kwds={'bins': 30, 'color': 'steelblue', 'alpha': 0.5})

# Rename axes
labels = ['OIL\n(USD/barrel)', 'CPI', 'USD\n(Index)', 'FED\n(%)', 'IND\n(Index)']
for i, (ax_row, label) in enumerate(zip(axes_sm, labels)):
    ax_row[0].set_ylabel(label, fontsize=9)
    axes_sm[-1][i].set_xlabel(label, fontsize=9)

plt.suptitle('Scatter matrix: Relationships between variable pairs', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. Log-Return transform & preliminary stationarity check

In [ ]:
# ── Calculate log-return ────────────────────────────────────────────────────────────
df_ret = pd.DataFrame(index=df_raw.index)
for col in ['OIL', 'CPI', 'USD', 'FED', 'IND']:
    df_ret[f'{col}_RET'] = np.log(df_raw[col] / df_raw[col].shift(1))

# FED contains zero values → treat separately with first difference
df_ret['FED_DIFF'] = df_raw['FED'].diff()

df_ret.dropna(inplace=True)

print('Log-return statistics:')
print(df_ret[['OIL_RET','CPI_RET','USD_RET','IND_RET']].describe().round(6))

In [ ]:
# ── Level vs Log-return comparison ───────────────────────────────────────────────
fig, axes = plt.subplots(5, 2, figsize=(16, 18))
fig.suptitle('Level vs Log-Return: Visual stationarity check', 
             fontsize=13, fontweight='bold')

ret_cols = ['OIL_RET', 'CPI_RET', 'USD_RET', 'FED_DIFF', 'IND_RET']

for i, (col, ret_col, (label, unit, color)) in enumerate(zip(
    ['OIL','CPI','USD','FED','IND'], ret_cols, var_labels.values()
)):
    # Level
    axes[i,0].plot(df_raw.index, df_raw[col], color=color, lw=1.2)
    axes[i,0].set_ylabel(f'{label}\n({unit})', fontsize=9)
    if i == 0:
        axes[i,0].set_title('Original series (Level) — usually non-stationary', fontsize=11, fontweight='bold')
    
    # Log-return
    ret_data = df_ret[ret_col]
    axes[i,1].plot(ret_data.index, ret_data, color=color, lw=0.9, alpha=0.8)
    axes[i,1].axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    axes[i,1].fill_between(ret_data.index, ret_data, 0,
                            where=ret_data > 0, color='green', alpha=0.15)
    axes[i,1].fill_between(ret_data.index, ret_data, 0,
                            where=ret_data < 0, color='red', alpha=0.15)
    ret_name = 'Diff' if 'DIFF' in ret_col else 'Log-Return'
    axes[i,1].set_ylabel(f'{ret_name}', fontsize=9)
    if i == 0:
        axes[i,1].set_title('Difference/Log-Return series — usually stationary', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Time series analysis – Rolling statistics

In [ ]:
# ── Rolling mean and std for oil price (12-month window) ─────────────────────────
window = 12
oil = df_raw['OIL']
roll_mean = oil.rolling(window).mean()
roll_std  = oil.rolling(window).std()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Oil price + rolling mean
ax1.plot(oil.index, oil.values, color='#1f77b4', lw=1, alpha=0.7, label='Actual oil price')
ax1.plot(roll_mean.index, roll_mean.values, color='red', lw=2, label=f'Rolling Mean ({window}T)')
ax1.fill_between(oil.index, 
                  roll_mean - 2*roll_std, roll_mean + 2*roll_std,
                  alpha=0.15, color='gray', label='±2σ (Bollinger Band)')
ax1.set_ylabel('Oil price (USD/barrel)')
ax1.set_title('Brent Oil: Rolling Mean ± 2σ (Bollinger Bands)', fontweight='bold')
ax1.legend(fontsize=9)

# Rolling std = volatility
ax2.plot(roll_std.index, roll_std.values, color='orange', lw=1.8)
ax2.fill_between(roll_std.index, roll_std.values, alpha=0.3, color='orange')
ax2.set_ylabel('Rolling Std (12T)')
ax2.set_title('Rolling 12-month volatility', fontweight='bold')

# Highlight high volatility periods
high_vol = roll_std > roll_std.quantile(0.75)
ax2.fill_between(roll_std.index, 0, roll_std.values,
                  where=high_vol, color='red', alpha=0.3, label='High volatility (>Q75)')
ax2.legend(fontsize=9)

for ax in [ax1, ax2]:
    for ev_name, (ev_date, ev_color) in events.items():
        ax.axvline(pd.Timestamp(ev_date), color=ev_color, lw=1.2, ls='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ── Rolling correlation: OIL vs other variables ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Rolling correlation (24 months): OIL vs other variables', 
             fontsize=13, fontweight='bold')

other_vars = [
    ('CPI', 'CPI Mỹ', '#ff7f0e'),
    ('USD', 'Chỉ số USD', '#2ca02c'),
    ('FED', 'Fed rate', '#d62728'),
    ('IND', 'Industrial production', '#9467bd'),
]

for ax, (col, label, color) in zip(axes.flat, other_vars):
    roll_corr = df_raw['OIL'].rolling(24).corr(df_raw[col])
    ax.plot(roll_corr.index, roll_corr.values, color=color, lw=1.5)
    ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(0.5, color='green', lw=0.8, ls=':', alpha=0.5)
    ax.axhline(-0.5, color='red', lw=0.8, ls=':', alpha=0.5)
    ax.fill_between(roll_corr.index, roll_corr.values, 0,
                     where=roll_corr > 0, color='green', alpha=0.15)
    ax.fill_between(roll_corr.index, roll_corr.values, 0,
                     where=roll_corr < 0, color='red', alpha=0.15)
    ax.set_ylim(-1, 1)
    ax.set_title(f'OIL ↔ {label}', fontweight='bold')
    ax.set_ylabel('Pearson r')
    
    # Event annotations
    for ev_name, (ev_date, ev_color) in events.items():
        ax.axvline(pd.Timestamp(ev_date), color=ev_color, lw=1, ls='--', alpha=0.6)

plt.tight_layout()
plt.show()

print('\n💡 Rolling correlation shows relationships change over time')
print('   Correlation is unstable → modeling structure should be chosen carefully')

## 11. Historical period analysis

In [ ]:
# ── Statistics by historical period ──────────────────────────────────────────────
periods = {
    'Normal (2006–2008)': ('2006-01-01', '2008-08-31'),
    'Financial crisis (2008–2009)': ('2008-09-01', '2009-06-30'),
    'Phục hồi (2009–2014)': ('2009-07-01', '2014-10-31'),
    'OPEC price drop (2014–2016)': ('2014-11-01', '2016-06-30'),
    'Stable period (2016–2020)': ('2016-07-01', '2020-02-29'),
    'COVID-19 (2020)': ('2020-03-01', '2020-12-31'),
    'Phục hồi COVID (2021–2022/01)': ('2021-01-01', '2022-01-31'),
    'Ukraine War (2022)': ('2022-02-01', '2022-12-31'),
    'Hậu chiến (2023–2025)': ('2023-01-01', '2025-12-31'),
}

rows = []
for period_name, (start, end) in periods.items():
    mask = (df_raw.index >= start) & (df_raw.index <= end)
    sub = df_raw.loc[mask, 'OIL']
    if len(sub) == 0:
        continue
    rows.append({
        'Period': period_name,
        'Months': len(sub),
        'OIL Mean': sub.mean(),
        'OIL Std': sub.std(),
        'OIL Min': sub.min(),
        'OIL Max': sub.max(),
        'OIL CV%': sub.std()/sub.mean()*100,
    })

df_periods = pd.DataFrame(rows).set_index('Period')
print('OIL STATISTICS BY HISTORICAL PERIOD')
print('='*80)
print(df_periods.round(2).to_string())
print('\nCV% = Coefficient of variation (higher = more unstable)')

In [ ]:
# ── Oil boxplot by period ────────────────────────────────────────────
period_data = []
period_names_short = []

for period_name, (start, end) in periods.items():
    mask = (df_raw.index >= start) & (df_raw.index <= end)
    sub = df_raw.loc[mask, 'OIL']
    if len(sub) == 0:
        continue
    period_data.append(sub.values)
    short = period_name.split('(')[0].strip()[:25]
    period_names_short.append(short)

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(period_data, patch_artist=True, notch=False,
                medianprops=dict(color='red', lw=2))

colors_period = plt.cm.tab10(np.linspace(0, 1, len(period_data)))
for patch, color in zip(bp['boxes'], colors_period):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_xticks(range(1, len(period_names_short)+1))
ax.set_xticklabels(period_names_short, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Brent oil price (USD/barrel)')
ax.set_title('Oil price distribution by historical period', 
             fontsize=13, fontweight='bold')
ax.axhline(df_raw['OIL'].mean(), color='navy', ls='--', lw=1.5, 
           alpha=0.7, label=f'Overall mean = {df_raw["OIL"].mean():.1f}')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 12. Seasonality analysis

In [ ]:
# ── Month-of-year analysis ────────────────────────────────────────────
df_raw['month'] = df_raw.index.month
df_raw['year']  = df_raw.index.year

month_names = ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Monthly boxplot
monthly_groups = [df_raw[df_raw['month'] == m]['OIL'].values for m in range(1, 13)]
bp = axes[0].boxplot(monthly_groups, patch_artist=True,
                      medianprops=dict(color='red', lw=2))
colors_m = plt.cm.coolwarm(np.linspace(0, 1, 12))
for patch, color in zip(bp['boxes'], colors_m):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0].set_xticklabels(month_names)
axes[0].set_ylabel('Oil price (USD/barrel)')
axes[0].set_title('Oil price distribution by month', fontweight='bold')
axes[0].axhline(df_raw['OIL'].mean(), color='navy', ls='--', lw=1.5, alpha=0.7)

# Heatmap of average oil price by year × month
pivot = df_raw.pivot_table(values='OIL', index='year', columns='month', aggfunc='mean')
pivot.columns = month_names
sns.heatmap(pivot, ax=axes[1], cmap='RdYlGn_r', 
            annot=True, fmt='.0f', annot_kws={'size': 8},
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'USD/barrel'})
axes[1].set_title('Average oil price heatmap: Year × Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Year')

plt.suptitle('Brent oil seasonality analysis', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

df_raw.drop(columns=['month', 'year'], inplace=True)  # remove temporary columns

## 13. ACF & PACF – Autocorrelation check

In [ ]:
def plot_acf_pacf(series, title, max_lag=36, ax_acf=None, ax_pacf=None):
    """Plot ACF and PACF manually (Yule-Walker)"""
    n = len(series)
    s = series - series.mean()
    c0 = np.dot(s, s) / n
    
    # ACF
    acf_vals = [1.0] + [np.dot(s[k:], s[:-k]) / (n * c0) for k in range(1, max_lag+1)]
    lags = np.arange(0, max_lag+1)
    ci = 1.96 / np.sqrt(n)
    
    ax_acf.bar(lags, acf_vals, color='#1f77b4', alpha=0.7, width=0.6)
    ax_acf.axhline(ci, color='red', ls='--', lw=1.2, alpha=0.7, label=f'±1.96/√n = ±{ci:.3f}')
    ax_acf.axhline(-ci, color='red', ls='--', lw=1.2, alpha=0.7)
    ax_acf.axhline(0, color='black', lw=0.8)
    ax_acf.set_title(f'ACF: {title}', fontsize=10)
    ax_acf.set_xlabel('Lag'); ax_acf.set_ylabel('Coefficient')
    ax_acf.legend(fontsize=8)
    ax_acf.set_xlim(-0.5, max_lag+0.5)
    
    # PACF (Yule-Walker)
    pacf_vals = [1.0]
    for k in range(1, max_lag+1):
        r = np.array([np.dot(s[j:], s[:-j]) / (n * c0) for j in range(1, k+1)])
        R = np.array([[np.dot(s[abs(i-j):], s[:n-abs(i-j)]) / (n * c0)
                       if i != j else 1.0
                       for j in range(1, k+1)] for i in range(1, k+1)])
        try:
            phi = np.linalg.solve(R, r)
            pacf_vals.append(phi[-1])
        except:
            pacf_vals.append(0)
    
    ax_pacf.bar(lags, pacf_vals, color='#ff7f0e', alpha=0.7, width=0.6)
    ax_pacf.axhline(ci, color='red', ls='--', lw=1.2, alpha=0.7)
    ax_pacf.axhline(-ci, color='red', ls='--', lw=1.2, alpha=0.7)
    ax_pacf.axhline(0, color='black', lw=0.8)
    ax_pacf.set_title(f'PACF: {title}', fontsize=10)
    ax_pacf.set_xlabel('Lag'); ax_pacf.set_ylabel('Coefficient')
    ax_pacf.set_xlim(-0.5, max_lag+0.5)

fig, axes = plt.subplots(4, 2, figsize=(14, 14))
fig.suptitle('ACF & PACF: Oil price (Level and Log-Return)', 
             fontsize=13, fontweight='bold')

plot_acf_pacf(df_raw['OIL'].values, 'OIL (Level)',
              ax_acf=axes[0,0], ax_pacf=axes[0,1])
plot_acf_pacf(df_ret['OIL_RET'].values, 'OIL Log-Return',
              ax_acf=axes[1,0], ax_pacf=axes[1,1])
plot_acf_pacf(df_ret['OIL_RET'].values**2, 'OIL Log-Return² (ARCH check)',
              ax_acf=axes[2,0], ax_pacf=axes[2,1])
plot_acf_pacf(np.abs(df_ret['OIL_RET'].values), 'OIL |Log-Return| (Volatility)',
              ax_acf=axes[3,0], ax_pacf=axes[3,1])

plt.tight_layout()
plt.show()

print('\n💡 Interpreting ACF/PACF:')
print('  - Level: ACF decays slowly → non-stationary series (trend present)')
print('  - Log-Return: ACF cuts off quickly → near stationary')
print('  - Log-Return²: significant ACF → ARCH effects present (volatility clustering)')

## 14. EDA summary & conclusions

In [ ]:
print('=' * 70)
print('EDA SUMMARY – BRENT OIL ANALYSIS DATASET')
print('=' * 70)

print('''
📌 1. CLEANED DATA
   • Time range: 2006 – 2025 (about 240 months)
   • 5 variables: OIL, CPI, USD, FED, IND
   • Frequency: Monthly (resampled from daily/monthly)
   • NaN handling: ffill(3) → bfill(3) → dropna

📌 2. DISTRIBUTION
   • OIL : Right skew (skew > 0), heavy tails → NOT normal (JB test)
   • CPI : Strong right skew due to long-term upward trend
   • USD : Relatively symmetric
   • FED : Bimodal (two peaks) due to strong rate policy shifts
   • IND : Slight left skew

📌 3. CORRELATION
   • OIL & CPI  : Strong positive correlation (same long-term trend)
   • OIL & USD  : Negative correlation (strong USD → cheaper oil overseas)
   • OIL & FED  : Complex, time-varying relationship
   • Rolling correlation shows the relationship is not stable over time

📌 4. STATIONARITY
   • Level : Non-stationary – ACF decays slowly, clear trend
   • Log-Return : Near stationary – suitable for modeling
   • Log-Return² : Autocorrelation exists → ARCH effect → consider GARCH

📌 5. OUTLIERS & MAJOR EVENTS
   • 2008–2009 : Oil crash due to global financial crisis
   • 2014–2016 : OPEC kept output high → price shock down
   • 2020/04   : Oil plunged to very low levels (even negative WTI) due to COVID
   • 2022/03   : Spike due to Russia–Ukraine war

📌 6. MODEL SUGGESTIONS
   • Use OIL log-return as dependent variable
   • Run ADF + KPSS tests to confirm stationarity
   • ARIMA / SARIMA for trend
   • GARCH / EGARCH for volatility clustering
   • Add dummy variables for major shocks (GFC, COVID, Ukraine)
''')

print('=' * 70)
print('✅ EDA complete – data ready for modeling')
print('=' * 70)

In [ ]:
# ── Save cleaned data ───────────────────────────────────────────────────
df_raw.to_csv('data_cleaned_level.csv')
df_ret.to_csv('data_cleaned_returns.csv')

print('✅ Saved:')
print('   data_cleaned_level.csv   — cleaned level data (240 months)')
print('   data_cleaned_returns.csv — computed log-return data')